# Test Rollout-finetuned Temperature Surrogate

This notebook tests the rollout-finetuned checkpoint on the virtual geometry case. It runs an autoregressive rollout, computes metrics, plots final-frame horizontal slices, and generates GIFs. The vertical GIF uses a fixed `y` plane, showing an `x-z` section along the main wind direction.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Image as IPyImage, display

# Locate this rollout folder whether the notebook is opened from
# temperature_field_bundle/ or from rollout_surrogate/.
cwd = Path.cwd().resolve()
if (cwd / 'test_virtual_geometry_rollout_surrogate.py').exists():
    ROLLOUT_DIR = cwd
elif (cwd / 'rollout_surrogate' / 'test_virtual_geometry_rollout_surrogate.py').exists():
    ROLLOUT_DIR = cwd / 'rollout_surrogate'
else:
    raise FileNotFoundError('Open this notebook from temperature_field_bundle or rollout_surrogate.')

if str(ROLLOUT_DIR) not in sys.path:
    sys.path.insert(0, str(ROLLOUT_DIR))

import test_virtual_geometry_rollout_surrogate as t

print('Rollout dir:', ROLLOUT_DIR)
print('Bundle root:', t.BUNDLE_ROOT)
print('Case dir:', t.CASE_DIR)
print('Output dir:', t.OUT_DIR)
print('Rollout checkpoint:', t.ROLLOUT_CHECKPOINT)

## Load Checkpoint and Case

In [ ]:
if not t.CASE_DIR.exists():
    raise FileNotFoundError(
        f'Missing virtual geometry case: {t.CASE_DIR}\n'
        'Run Test_Virtual_Geometry_3D_UNet_Surrogate.ipynb once to generate the physical reference case.'
    )

checkpoint_path = t.ROLLOUT_CHECKPOINT if t.ROLLOUT_CHECKPOINT.exists() else t.FALLBACK_CHECKPOINT
if checkpoint_path == t.FALLBACK_CHECKPOINT:
    print('Rollout checkpoint not found; falling back to one-step checkpoint for comparison.')

checkpoint = torch.load(checkpoint_path, map_location='cpu')
config = checkpoint.get('config', {})
t.CORRECTION_ITERATIONS = int(config.get('correction_iterations', t.CORRECTION_ITERATIONS))
stats = t.SurrogateStats(**checkpoint['stats'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = t.load_model_from_checkpoint(checkpoint, device)

case = t.apply_z_crop(t.Temperature3DCase(t.CASE_DIR))

print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Loaded checkpoint:', checkpoint_path)
print('Checkpoint epoch:', checkpoint.get('epoch', 'unknown'))
print('Z crop:', t.Z_CROP)
print('Temperature shape:', case.temperature.shape)
print('Velocity shape:', case.u.shape)

## Autoregressive Rollout

In [ ]:
pred_seq, true_seq, valid_mask, frame_ids = t.autoregressive_rollout(case, model, stats, device)
err_seq = pred_seq - true_seq

frame_bias_c = [float(np.mean(frame_err[valid_mask])) for frame_err in err_seq]
frame_mae_c = [float(np.mean(np.abs(frame_err[valid_mask]))) for frame_err in err_seq]
frame_rmse_c = [float(np.sqrt(np.mean(frame_err[valid_mask] ** 2))) for frame_err in err_seq]

metrics = {
    'checkpoint': str(checkpoint_path),
    'case_dir': str(t.CASE_DIR),
    'mode': 'autoregressive_rollout',
    'use_next_velocity': t.USE_NEXT_VELOCITY,
    'correction_iterations': t.CORRECTION_ITERATIONS,
    'z_crop': t.Z_CROP,
    'patch_size': list(t.PATCH_SIZE),
    'use_cool_surface_channels': t.USE_COOL_SURFACE_CHANNELS,
    'use_cool_surface_relaxation': t.USE_COOL_SURFACE_RELAXATION,
    'cool_surface_relaxation': t.COOL_SURFACE_RELAXATION,
    'open_ground_relaxation': t.OPEN_GROUND_RELAXATION,
    'cool_surface_vertical_decay_layers': t.COOL_SURFACE_VERTICAL_DECAY_LAYERS,
    'in_channels': t.input_channel_count(t.HISTORY_STEPS),
    'frames': [int(v) for v in frame_ids],
    'first_predicted_frame': int(frame_ids[0]),
    'first_step_bias_c': float(np.mean(err_seq[0][valid_mask])),
    'first_step_mae_c': float(np.mean(np.abs(err_seq[0][valid_mask]))),
    'first_step_rmse_c': float(np.sqrt(np.mean(err_seq[0][valid_mask] ** 2))),
    'final_frame': int(frame_ids[-1]),
    'final_bias_c': float(np.mean(err_seq[-1][valid_mask])),
    'final_mae_c': float(np.mean(np.abs(err_seq[-1][valid_mask]))),
    'final_rmse_c': float(np.sqrt(np.mean(err_seq[-1][valid_mask] ** 2))),
    'rollout_bias_c': float(np.mean(err_seq[:, valid_mask])),
    'rollout_mae_c': float(np.mean(np.abs(err_seq[:, valid_mask]))),
    'rollout_rmse_c': float(np.sqrt(np.mean(err_seq[:, valid_mask] ** 2))),
    'coldest_step_bias_c': float(np.min(frame_bias_c)),
    'warmest_step_bias_c': float(np.max(frame_bias_c)),
    'max_abs_step_bias_c': float(np.max(np.abs(frame_bias_c))),
    'frame_bias_c': frame_bias_c,
    'frame_mae_c': frame_mae_c,
    'frame_rmse_c': frame_rmse_c,
}
metrics_path = t.OUT_DIR / 'virtual_geometry_rollout_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
print('Saved metrics:', metrics_path)

## Frame 24/48 Horizontal Panels

In [ ]:
velocity_dir = case._velocity_cache_dir()
building_mask_path = velocity_dir / 'building_mask_2d.npy'
building_mask_2d = np.load(building_mask_path).astype(bool) if building_mask_path.exists() else np.any(case.solid_mask, axis=0)
save_final_panel_supports_filename = 'out_filename' in t.save_final_panel.__code__.co_varnames

requested_panel_frames = [24, 48]
available_frames = [int(v) for v in frame_ids]
saved_panel_paths = []
for requested_frame in requested_panel_frames:
    nearest_frame = min(available_frames, key=lambda frame: abs(frame - requested_frame))
    seq_idx = available_frames.index(nearest_frame)
    out_name = f'frame_{nearest_frame:03d}_horizontal_panels.png'
    panel_path = t.OUT_DIR / out_name
    if save_final_panel_supports_filename:
        t.save_final_panel(
            pred_seq[seq_idx],
            true_seq[seq_idx],
            valid_mask,
            nearest_frame,
            building_mask_2d,
            out_filename=out_name,
        )
    else:
        t.save_final_panel(pred_seq[seq_idx], true_seq[seq_idx], valid_mask, nearest_frame, building_mask_2d)
        default_panel_path = t.OUT_DIR / 'final_frame_horizontal_panels.png'
        default_panel_path.replace(panel_path)
    saved_panel_paths.append(panel_path)
    print('Saved panel:', panel_path)
    display(IPyImage(filename=str(panel_path)))

## GIF Animations

The horizontal GIF shows a fixed `z` slice. The vertical GIF now uses a fixed `y` plane, so it shows an `x-z` section along the main wind direction.

In [ ]:
t.save_gifs(pred_seq, true_seq, valid_mask, frame_ids, building_mask_2d)

z_idx = min(4, true_seq.shape[1] - 1)
y_idx = true_seq.shape[2] // 2
horizontal_gif_path = t.OUT_DIR / f'horizontal_z{z_idx:02d}_rollout_finetuned.gif'
vertical_gif_path = t.OUT_DIR / f'vertical_y{y_idx:03d}_xz_rollout_finetuned.gif'

print('Saved horizontal GIF:', horizontal_gif_path)
display(IPyImage(filename=str(horizontal_gif_path)))

print('Saved vertical x-z GIF:', vertical_gif_path)
display(IPyImage(filename=str(vertical_gif_path)))

## Optional: Plot Error Through Time

In [ ]:
frame_mae = np.asarray([np.mean(np.abs(err_seq[i][valid_mask])) for i in range(err_seq.shape[0])], dtype=np.float32)
frame_bias = np.asarray([np.mean(err_seq[i][valid_mask]) for i in range(err_seq.shape[0])], dtype=np.float32)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(frame_ids, frame_mae, marker='o')
axes[0].set_title('Autoregressive rollout MAE')
axes[0].set_xlabel('frame')
axes[0].set_ylabel('deg C')
axes[0].grid(True, alpha=0.3)
axes[1].plot(frame_ids, frame_bias, marker='o')
axes[1].axhline(0.0, color='black', linewidth=0.8)
axes[1].set_title('Autoregressive rollout bias')
axes[1].set_xlabel('frame')
axes[1].set_ylabel('deg C')
axes[1].grid(True, alpha=0.3)
plt.show()